# Exploracion de modelos — Customer Churn

Objetivo: estimar si un cliente se va (`Churn = Yes`) **antes** de que se vaya.

El error que mas duele es el **falso negativo**: el modelo dice que se queda y el cliente se va. Perdemos la chance de retenerlo. Un falso positivo (contactar de mas) es mas barato.

Por eso **no elegimos por accuracy**. Con ~73.6% de clientes estables, un modelo que nunca predice abandono ya “acierta” 3 de cada 4.

Criterio de seleccion: **recall de churn** (menos FN), con **ROC-AUC** competitivo y F1 como equilibrio. Precision se mira, no se maximiza a ciegas.

## 1. Datos y particion

Misma fuente que el EDA. `customerID` no entra como predictor. `TotalCharges` a veces viene como texto: lo pasamos a numerico; los vacios (tenure 0) quedan NaN y los imputamos adentro del pipeline, nunca a mano sobre todo el dataset.

Split estratificado 80/20, `random_state=42`. El test no se usa para elegir hiperparametros: solo para comparar al final.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

SEMILLA = 42
TAMANIO_TEST = 0.20

df = pd.read_csv("../data/raw/customer_churn_historical.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["Churn", "customerID"])

cols_num = X.select_dtypes(include="number").columns.tolist()
cols_cat = X.select_dtypes(include="object").columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TAMANIO_TEST, random_state=SEMILLA, stratify=y
)

print(f"dataset: {df.shape[0]} filas")
print(f"train: {len(X_train)}  |  test: {len(X_test)}")
print(f"churn train: {y_train.mean():.3f}  |  churn test: {y_test.mean():.3f}")
print(f"numericas: {cols_num}")
print(f"categoricas: {len(cols_cat)} columnas")

## 2. Preprocessing

Un solo `ColumnTransformer` para todos los modelos:

- numericas: mediana + `StandardScaler` (logreg lo necesita; a los arboles no les molesta)
- categoricas: moda + one-hot (`handle_unknown="ignore"` para categorias nuevas en inferencia)

Va **adentro** del `Pipeline` para que las estadisticas se calculen solo en train. Si imputaramos o escalaramos sobre el CSV completo, estariamos filtrando informacion del test.

In [ ]:
def armar_pipeline(estimador):
    pre = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                cols_num,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                cols_cat,
            ),
        ]
    )
    return Pipeline([("pre", pre), ("modelo", estimador)])


modelos = {
    "dummy_frecuente": DummyClassifier(strategy="most_frequent"),
    "dummy_estratificado": DummyClassifier(strategy="stratified", random_state=SEMILLA),
    "logreg": LogisticRegression(max_iter=1000, C=1.0, random_state=SEMILLA),
    "logreg_bal": LogisticRegression(
        max_iter=1000, C=0.5, class_weight="balanced", random_state=SEMILLA
    ),
    "rf": RandomForestClassifier(n_estimators=100, random_state=SEMILLA, n_jobs=-1),
    "rf_bal": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight="balanced",
        random_state=SEMILLA,
        n_jobs=-1,
    ),
    "gb": GradientBoostingClassifier(n_estimators=100, random_state=SEMILLA),
}

## 3. Comparacion

Siete corridas: dos baselines, lineal con y sin balanceo, dos random forests y un boosting.

Primero **validacion cruzada estratificada en train** (5 folds): ahi se ve si una familia es estable, sin tocar el test.
Despues entrenamos en todo el train y medimos el test una sola vez, umbral 0.5.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
scoring = {
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

cv_filas = []
for nombre, estimador in modelos.items():
    scores = cross_validate(
        armar_pipeline(estimador),
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
    )
    cv_filas.append(
        {
            "modelo": nombre,
            "recall_cv": scores["test_recall"].mean(),
            "f1_cv": scores["test_f1"].mean(),
            "auc_cv": scores["test_roc_auc"].mean(),
        }
    )

tabla_cv = pd.DataFrame(cv_filas).sort_values("recall_cv", ascending=False).round(3)
tabla_cv

In [ ]:
ajustes = {}
filas_test = []

for nombre, estimador in modelos.items():
    pipe = armar_pipeline(estimador)
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    ajustes[nombre] = {"pipe": pipe, "pred": pred, "proba": proba}
    filas_test.append(
        {
            "modelo": nombre,
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
            "roc_auc": roc_auc_score(y_test, proba),
            "FN": int(fn),
            "FP": int(fp),
            "TP": int(tp),
        }
    )

tabla = pd.DataFrame(filas_test).sort_values("recall", ascending=False)
tabla_mostrar = tabla.copy()
for col in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    tabla_mostrar[col] = tabla_mostrar[col].round(3)
tabla_mostrar

Lectura rapida de la tabla:

- **dummy_frecuente** confirma el truco de accuracy: ~0.74 y **recall 0**. Cero detecciones.
- **logreg** y **gb** ganan accuracy/precision, pero dejan escapar mas de la mitad de los abandonos.
- **logreg_bal** y **rf_bal** son los unicos que empujan recall de verdad. El lineal lo hace con AUC igual de alto y menos complejidad.

El ranking de CV en train y el ranking de test coinciden en lo importante: hay que balancear la clase o el modelo se duerme en la mayoria.

## 4. Curvas ROC y Precision-Recall

Accuracy es un punto (umbral 0.5). Las curvas miran **todo** el ranking de probabilidades.
ROC-AUC dice si el modelo ordena bien riesgo. PR es mas honesta con desbalance: ahi se ve el costo de subir recall.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for nombre, pack in ajustes.items():
    fpr, tpr, _ = roc_curve(y_test, pack["proba"])
    prec, rec, _ = precision_recall_curve(y_test, pack["proba"])
    auc = roc_auc_score(y_test, pack["proba"])
    axes[0].plot(fpr, tpr, label=f"{nombre} ({auc:.2f})")
    axes[1].plot(rec, prec, label=nombre)

axes[0].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[0].set_xlabel("Falsos positivos")
axes[0].set_ylabel("Recall")
axes[0].set_title("ROC (test)")
axes[0].legend(fontsize=8)

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color="gray", linestyle="--", linewidth=1, label=f"base churn ({baseline_pr:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall (test)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Matrices de confusion

Orden sklearn: `[[TN, FP], [FN, TP]]`.
La celda de abajo a la izquierda es el FN: clientes que se van y el modelo no aviso.

In [ ]:
claves = ["dummy_frecuente", "logreg", "logreg_bal", "rf", "rf_bal", "gb"]
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for ax, nombre in zip(axes.ravel(), claves):
    cm = confusion_matrix(y_test, ajustes[nombre]["pred"])
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["pred No", "pred Yes"],
        yticklabels=["real No", "real Yes"],
    )
    fn = cm[1, 0]
    ax.set_title(f"{nombre}  |  FN={fn}")

plt.tight_layout()
plt.show()

## 6. El candidato tiene que coincidir con el EDA

Si `logreg_bal` es el modelo, sus coeficientes deberian empujar churn en las mismas zonas que vimos en el EDA: contrato mes a mes, fibra, poco tenure, pago electronico. Si saliera otra historia, el modelo estaria aprendiendo ruido.

In [ ]:
pipe_cand = ajustes["logreg_bal"]["pipe"]
nombres = pipe_cand.named_steps["pre"].get_feature_names_out()
coefs = pipe_cand.named_steps["modelo"].coef_[0]

imp = pd.DataFrame({"feature": nombres, "coef": coefs})
imp["abs"] = imp["coef"].abs()
top = imp.sort_values("abs", ascending=False).head(15).sort_values("coef")

colores = np.where(top["coef"] > 0, "indianred", "steelblue")
plt.figure(figsize=(8, 6))
plt.barh(top["feature"], top["coef"], color=colores)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("logreg_bal — coeficientes (rojo = mas churn)")
plt.xlabel("coeficiente")
plt.tight_layout()
plt.show()

top[["feature", "coef"]].round(3)

## 7. El umbral 0.5 no es sagrado

`predict()` corta en 0.5. En churn eso es una decision de negocio, no una ley de sklearn.
Bajar el umbral sube recall (menos FN) y tira abajo precision (mas contactos al pedo).

Barrido sobre las probabilidades de `logreg_bal` en test. Las bandas LOW / MEDIUM / HIGH (0.35 y 0.55) son una propuesta operativa, no un hiperparametro del algoritmo.

In [ ]:
proba_c = ajustes["logreg_bal"]["proba"]
umbrales = np.round(np.arange(0.30, 0.65, 0.05), 2)

filas_u = []
for u in umbrales:
    pred_u = (proba_c >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_u).ravel()
    filas_u.append(
        {
            "umbral": u,
            "precision": precision_score(y_test, pred_u, zero_division=0),
            "recall": recall_score(y_test, pred_u, zero_division=0),
            "f1": f1_score(y_test, pred_u, zero_division=0),
            "FN": int(fn),
            "FP": int(fp),
        }
    )

tabla_u = pd.DataFrame(filas_u).round(3)
display(tabla_u)

plt.figure(figsize=(8, 4))
plt.plot(tabla_u["umbral"], tabla_u["recall"], marker="o", label="recall")
plt.plot(tabla_u["umbral"], tabla_u["precision"], marker="o", label="precision")
plt.plot(tabla_u["umbral"], tabla_u["f1"], marker="o", label="f1")
plt.axvline(0.35, color="gray", linestyle="--", linewidth=1)
plt.axvline(0.55, color="gray", linestyle="--", linewidth=1)
plt.title("logreg_bal — trade-off segun umbral")
plt.xlabel("umbral")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Decision

**Candidato: `logreg_bal`** — `LogisticRegression(C=0.5, class_weight="balanced")` dentro del mismo pipeline de preprocessing.

Por que ese y no otro:

1. **Negocio.** Mejor recall de los lineales, FN claramente mas bajos que logreg/gb/rf default. Preferimos contactar de mas antes que perder al cliente.
2. **Discriminacion.** ROC-AUC al mismo nivel que logreg sin balancear (~0.81). El balanceo no “inventa” senal: cambia el punto de corte efectivo.
3. **Interpretable.** Los coeficientes reproducen el EDA (contrato corto, fibra, poca antiguedad). Eso importa si hay que defender el modelo.
4. **Simple de servir.** Lineal + pipeline sklearn; no hace falta un ensemble mas pesado si no gana recall.

Que no se elige:

- **dummy:** piso, no producto.
- **logreg / gb / rf default:** accuracy linda, cobertura de churn pobre.
- **rf_bal:** cerca en F1, peor o similar recall, mas opaco. No justifica la complejidad en esta etapa.

Trade-off aceptado: baja accuracy y precision a cambio de detectar abandonos. El umbral se puede mover despues (bandas 0.35 / 0.55) cuando existan costos reales de contacto vs. perdida de cliente.